In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
ratings = pd.read_csv("Ratings.csv", sep=",", quotechar='"', encoding='latin-1', low_memory=False)
ratings = ratings[ratings['Book-Rating'] > 0]  # filter out zero ratings

users = pd.read_csv("Users.csv", sep=",", quotechar='"', encoding='latin-1', low_memory=False)

books = pd.read_csv("Books.csv", sep=",", quotechar='"', encoding='latin-1', low_memory=False)

In [8]:
print(ratings.head())

   User-ID        ISBN  Book-Rating
1   276726  0155061224            5
3   276729  052165615X            3
4   276729  0521795028            6
6   276736  3257224281            8
7   276737  0600570967            6


In [10]:
print(users.head())

   User-ID                            Location   Age
0        1                  nyc, new york, usa   NaN
1        2           stockton, california, usa  18.0
2        3     moscow, yukon territory, russia   NaN
3        4           porto, v.n.gaia, portugal  17.0
4        5  farnborough, hants, united kingdom   NaN


In [12]:
print(books.head())

         ISBN                                         Book-Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

            Book-Author Year-Of-Publication                   Publisher  \
0    Mark P. O. Morford                2002     Oxford University Press   
1  Richard Bruce Wright                2001       HarperFlamingo Canada   
2          Carlo D'Este                1991             HarperPerennial   
3      Gina Bari Kolata                1999        Farrar Straus Giroux   
4       E. J. W. Barber                1999  W. W. Norton &amp; Company   

                                         Image-URL-S  \
0  http://images.amazon.com/images/P/0195153448.0...   
1  http://images.amazon.com/

In [14]:
active_users = ratings['User-ID'].value_counts()[ratings['User-ID'].value_counts() >= 10].index
popular_books = ratings['ISBN'].value_counts()[ratings['ISBN'].value_counts() >= 10].index

filtered_ratings = ratings[
    ratings['User-ID'].isin(active_users) & ratings['ISBN'].isin(popular_books)
]

In [16]:
print("Null Values in ratings",ratings.isnull().sum())

Null Values in ratings User-ID        0
ISBN           0
Book-Rating    0
dtype: int64


In [18]:
train_data, test_data=train_test_split(filtered_ratings,test_size=0.25, random_state=42)
train_data=train_data.reset_index(drop=True)
test_data=test_data.reset_index(drop=True)

In [20]:
print("Train set size", train_data.shape)
print("Test set size", test_data.shape)

Train set size (67917, 3)
Test set size (22639, 3)


In [22]:
print(train_data.head())

   User-ID        ISBN  Book-Rating
0   250709  0425165701            5
1   141089  0671534645            8
2   129851  0345337662            7
3   211426  0060976241            9
4   254196  0449218473            3


In [26]:
test_data = test_data[
    test_data['User-ID'].isin(train_data['User-ID']) &
    test_data['ISBN'].isin(train_data['ISBN'])
].reset_index(drop=True)

In [28]:
user_item_matrix=train_data.pivot_table(index='User-ID', columns='ISBN', values='Book-Rating')
user_item_matrix_filled=user_item_matrix.fillna(0)

In [30]:
item_user_matrix=user_item_matrix_filled.T
item_similarity=cosine_similarity(item_user_matrix)

In [32]:
item_similarity_df = pd.DataFrame(item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index)

In [38]:
from tqdm import tqdm
import numpy as np

In [40]:
def predict_rating(user_id, isbn, user_item_matrix, similarity_df, k=5):
    if isbn not in similarity_df.index or user_id not in user_item_matrix.index:
        return np.nan  # cannot predict

    # Books the user has rated
    user_ratings = user_item_matrix.loc[user_id]
    rated_books = user_ratings[user_ratings > 0]

    # Similarities to other books
    similar_books = similarity_df[isbn].drop(isbn, errors='ignore')
    common_books = similar_books[rated_books.index.intersection(similar_books.index)]

    if common_books.empty:
        return np.nan

    # Top-k similar books
    top_k = common_books.sort_values(ascending=False).head(k)
    top_k_ratings = rated_books[top_k.index]

    # Weighted average
    numerator = np.dot(top_k.values, top_k_ratings.values)
    denominator = np.sum(np.abs(top_k.values))
    if denominator == 0:
        return np.nan
    return numerator / denominator

def predict_test_ratings(test_df, user_item_matrix, similarity_df, k=5):
    predictions = []
    actuals = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        user = row['User-ID']
        isbn = row['ISBN']
        actual_rating = row['Book-Rating']
        predicted_rating = predict_rating(user, isbn, user_item_matrix, similarity_df, k)

        if not np.isnan(predicted_rating):
            predictions.append(predicted_rating)
            actuals.append(actual_rating)

    return np.array(predictions), np.array(actuals)

def mean_absolute_difference(predictions, actuals):
    return np.mean(np.abs(predictions - actuals))

for k in [5, 10, 15, 20, 50, 100]:
    preds, acts = predict_test_ratings(test_data, user_item_matrix_filled, item_similarity_df, k)
    mad = mean_absolute_difference(preds, acts)
    print(f"k = {k}, MAD = {mad:.4f}")

100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 876.19it/s]


k = 5, MAD = 1.2460


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 872.28it/s]


k = 10, MAD = 1.2343


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 864.03it/s]


...

k = 20, MAD = 1.2318


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 863.51it/s]


k = 50, MAD = 1.2328


100%|██████████████████████████████████████████████████████████| 22381/22381 [00:25<00:00, 862.37it/s]

k = 100, MAD = 1.2327


In [42]:
sample_ratios = [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
k_values = [5, 10, 15, 20, 50, 100]

results = []

for ratio in sample_ratios:
    print(f"\n--- Sample Ratio: {int(ratio*100)}% ---")
    
    train_data, test_data = train_test_split(filtered_ratings, test_size=(1 - ratio), random_state=42)
    train_data = train_data.reset_index(drop=True)
    test_data = test_data.reset_index(drop=True)

    # Keep only test entries with users and books present in train
    test_data = test_data[
        test_data['User-ID'].isin(train_data['User-ID']) &
        test_data['ISBN'].isin(train_data['ISBN'])
    ].reset_index(drop=True)

    # User-item matrix for training
    user_item_matrix = train_data.pivot_table(index='User-ID', columns='ISBN', values='Book-Rating')
    user_item_matrix_filled = user_item_matrix.fillna(0)

    # Compute item similarity
    item_user_matrix = user_item_matrix_filled.T
    item_similarity = cosine_similarity(item_user_matrix)
    item_similarity_df = pd.DataFrame(item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index)

    for k in k_values:
        print(f"Evaluating for k = {k}...")
        preds, acts = predict_test_ratings(test_data, user_item_matrix_filled, item_similarity_df, k)
        mad = mean_absolute_difference(preds, acts)
        print(f"Sample Ratio = {int(ratio*100)}%, k = {k}, MAD = {mad:.4f}")
        results.append((int(ratio * 100), k, mad))



--- Sample Ratio: 60% ---
Evaluating for k = 5...


100%|██████████████████████████████████████████████████████████| 35618/35618 [00:40<00:00, 883.37it/s]


Sample Ratio = 60%, k = 5, MAD = 1.2629
Evaluating for k = 10...


100%|██████████████████████████████████████████████████████████| 35618/35618 [00:40<00:00, 881.22it/s]


Sample Ratio = 60%, k = 10, MAD = 1.2518
Evaluating for k = 15...


...

Sample Ratio = 90%, k = 20, MAD = 1.2033
Evaluating for k = 50...


100%|████████████████████████████████████████████████████████████| 8984/8984 [00:10<00:00, 851.98it/s]


Sample Ratio = 90%, k = 50, MAD = 1.2049
Evaluating for k = 100...


100%|████████████████████████████████████████████████████████████| 8984/8984 [00:10<00:00, 860.49it/s]

Sample Ratio = 90%, k = 100, MAD = 1.2059


In [44]:
results_df = pd.DataFrame(results, columns=['Sample_Ratio', 'K', 'MAD'])
print(results_df)

    Sample_Ratio    K       MAD
0             60    5  1.262930
1             60   10  1.251835
2             60   15  1.250616
3             60   20  1.250516
4             60   50  1.249909
5             60  100  1.249765
6             65    5  1.257688
7             65   10  1.246388
8             65   15  1.244726
9             65   20  1.244454
10            65   50  1.244086
11            65  100  1.243858
12            70    5  1.252841
13            70   10  1.242396
14            70   15  1.239374
15            70   20  1.239352
16            70   50  1.239556
17            70  100  1.239337
18            75    5  1.246011
19            75   10  1.234270
20            75   15  1.232509
21            75   20  1.231796
22            75   50  1.232768
23            75  100  1.232715
24            80    5  1.237355
25            80   10  1.227783
26            80   15  1.226269
27            80   20  1.225114
28            80   50  1.226485
29            80  100  1.226379
30      